In [3]:
import pandas as pd
import os
import numpy as np

BASE_DIR = os.path.expanduser("~/projects/nhs-ae-waiting-times")
SILVER_DIR = os.path.join(BASE_DIR, "data/silver")
GOLD_DIR = os.path.join(BASE_DIR, "data/gold")

# Load Silver
df = pd.read_parquet(os.path.join(SILVER_DIR, "silver_ae_providers.parquet"))
national_totals = pd.read_parquet(os.path.join(SILVER_DIR, "silver_ae_national_totals.parquet"))

print(f"Silver loaded: {df.shape}")
print(f"Date range: {df['period_date'].min().strftime('%b %Y')} → {df['period_date'].max().strftime('%b %Y')}")
print(f"\nProvider types:\n{df['provider_type'].value_counts().to_string()}")

Silver loaded: (7238, 25)
Date range: Apr 2022 → Mar 2025

Provider types:
provider_type
NHS_TRUST        5457
UTC_COMMUNITY     901
GP_PRACTICE       502
INDEPENDENT       307
NHS_OTHER          71


In [3]:
# ── GOLD TABLE 1: Provider-level monthly metrics ─────────────────

gold = df.copy()

# Total attendances (all types, excluding booked)
gold['total_att'] = gold['att_type1'] + gold['att_type2'] + gold['att_other']

# Total breaches (all types)
gold['total_over_4hr'] = gold['over_4hr_type1'] + gold['over_4hr_type2'] + gold['over_4hr_other']

# Total seen within 4hrs
gold['total_within_4hr'] = gold['total_att'] - gold['total_over_4hr']

# 4hr performance — Type 1 only (the real NHS headline target)
gold['perf_4hr_type1'] = np.where(
    gold['att_type1'] > 0,
    ((gold['att_type1'] - gold['over_4hr_type1']) / gold['att_type1'] * 100).round(1),
    np.nan  # Don't calculate % for providers with no Type 1 A&E
)

# 4hr performance — all types combined
gold['perf_4hr_all'] = np.where(
    gold['total_att'] > 0,
    (gold['total_within_4hr'] / gold['total_att'] * 100).round(1),
    np.nan
)

# Total emergency admissions
gold['total_emerg_admissions'] = (
    gold['emerg_admit_type1'] +
    gold['emerg_admit_type2'] +
    gold['emerg_admit_other'] +
    gold['emerg_admit_non_ae']
)

# 95% target met flag (Type 1 only)
gold['target_95_met'] = gold['perf_4hr_type1'] >= 95.0

# 12hr corridor care rate (per 100 attendances)
gold['dta_12hr_rate'] = np.where(
    gold['total_att'] > 0,
    (gold['dta_12hr_plus'] / gold['total_att'] * 100).round(2),
    np.nan
)

# Select final Gold columns
gold_providers = gold[[
    'period_date', 'org_code', 'org_name', 'parent_org', 'provider_type',
    'has_type1_ae', 'total_att', 'att_type1', 'att_type2', 'att_other',
    'total_over_4hr', 'over_4hr_type1', 'total_within_4hr',
    'perf_4hr_type1', 'perf_4hr_all', 'target_95_met',
    'dta_4_12hr', 'dta_12hr_plus', 'dta_12hr_rate',
    'total_emerg_admissions', 'emerg_admit_type1'
]].copy()

print(f"Gold providers shape: {gold_providers.shape}")
print(f"\nSample — NHS Trusts with Type 1 A&E (performance metrics):")
sample = gold_providers[
    (gold_providers['provider_type'] == 'NHS_TRUST') &
    (gold_providers['has_type1_ae'] == True) &
    (gold_providers['period_date'] == '2025-03-01')
][['org_name', 'att_type1', 'perf_4hr_type1', 'target_95_met', 'dta_12hr_plus']].head(10)
print(sample.to_string())

Gold providers shape: (7238, 21)

Sample — NHS Trusts with Type 1 A&E (performance metrics):
                                                   org_name  att_type1  perf_4hr_type1  target_95_met  dta_12hr_plus
6264            TORBAY AND SOUTH DEVON NHS FOUNDATION TRUST       6650            55.2          False             45
6265                 BARNSLEY HOSPITAL NHS FOUNDATION TRUST       9281            80.2          False              0
6267       CALDERDALE AND HUDDERSFIELD NHS FOUNDATION TRUST      16307            81.8          False             20
6273            HARROGATE AND DISTRICT NHS FOUNDATION TRUST       5137            74.4          False             23
6274                 SURREY AND SUSSEX HEALTHCARE NHS TRUST      10472            64.3          False            688
6276                    EAST LANCASHIRE HOSPITALS NHS TRUST      13340            66.6          False           1105
6282     SOUTH TYNESIDE AND SUNDERLAND NHS FOUNDATION TRUST      11463            58.0  

In [5]:
# ── GOLD TABLE 2: National monthly aggregates ─────────────────────

# Use only NHS Trusts for the headline national figure
# (this is how NHS England calculates the published national performance)
nhs_trusts = gold_providers[gold_providers['provider_type'] == 'NHS_TRUST'].copy()

national = nhs_trusts.groupby('period_date').agg(
    total_att=('total_att', 'sum'),
    att_type1=('att_type1', 'sum'),
    total_over_4hr=('total_over_4hr', 'sum'),
    over_4hr_type1=('over_4hr_type1', 'sum'),
    total_within_4hr=('total_within_4hr', 'sum'),
    dta_12hr_plus=('dta_12hr_plus', 'sum'),
    total_emerg_admissions=('total_emerg_admissions', 'sum'),
    trusts_reporting=('org_code', 'count')
).reset_index()

# Calculate national performance rates
national['perf_4hr_type1'] = (
    (national['att_type1'] - national['over_4hr_type1']) /
    national['att_type1'] * 100
).round(1)

national['perf_4hr_all'] = (
    national['total_within_4hr'] /
    national['total_att'] * 100
).round(1)

national['target_95_met'] = national['perf_4hr_type1'] >= 95.0

national['dta_12hr_rate_per_100'] = (
    national['dta_12hr_plus'] / national['total_att'] * 100
).round(2)

# Sort chronologically
national = national.sort_values('period_date').reset_index(drop=True)

print("National monthly time series:")
print(national[['period_date', 'att_type1', 'perf_4hr_type1', 
                 'target_95_met', 'dta_12hr_plus', 'trusts_reporting']].to_string())

National monthly time series:
   period_date  att_type1  perf_4hr_type1  target_95_met  dta_12hr_plus  trusts_reporting
0   2022-04-01    1279545            63.6          False          24179               154
1   2022-05-01    1402115            64.6          False          19657               154
2   2022-06-01    1367904            63.5          False          22017               154
3   2022-07-01    1365647            62.1          False          29139               154
4   2022-08-01    1276713            63.0          False          28689               151
5   2022-09-01    1274680            61.8          False          32749               152
6   2022-10-01    1369912            60.0          False          43783               153
7   2022-11-01    1365415            59.8          False          37853               153
8   2022-12-01    1410227            55.6          False          54573               153
9   2023-01-01    1215916            62.9          False          4272

In [7]:
# ── GOLD TABLE 3: Regional monthly aggregates ─────────────────────

regional = nhs_trusts.groupby(['period_date', 'parent_org']).agg(
    total_att=('total_att', 'sum'),
    att_type1=('att_type1', 'sum'),
    over_4hr_type1=('over_4hr_type1', 'sum'),
    total_within_4hr=('total_within_4hr', 'sum'),
    dta_12hr_plus=('dta_12hr_plus', 'sum'),
    total_emerg_admissions=('total_emerg_admissions', 'sum'),
    trusts_reporting=('org_code', 'count')
).reset_index()

regional['perf_4hr_type1'] = (
    (regional['att_type1'] - regional['over_4hr_type1']) /
    regional['att_type1'] * 100
).round(1)

regional['perf_4hr_all'] = (
    regional['total_within_4hr'] /
    regional['total_att'] * 100
).round(1)

regional = regional.sort_values(['period_date', 'parent_org']).reset_index(drop=True)

print("Regional breakdown — March 2025:")
march_regional = regional[regional['period_date'] == '2025-03-01']
print(march_regional[['parent_org', 'att_type1', 'perf_4hr_type1', 
                        'dta_12hr_plus', 'trusts_reporting']].to_string())

Regional breakdown — March 2025:
                               parent_org  att_type1  perf_4hr_type1  dta_12hr_plus  trusts_reporting
245           NHS ENGLAND EAST OF ENGLAND     156695            60.7           3871                14
246                    NHS ENGLAND LONDON     234246            63.0           9251                21
247                  NHS ENGLAND MIDLANDS     267799            58.6          10994                26
248  NHS ENGLAND NORTH EAST AND YORKSHIRE     223129            62.7           3376                24
249                NHS ENGLAND NORTH WEST     199093            57.9           9787                25
250                NHS ENGLAND SOUTH EAST     213014            62.6           5700                23
251                NHS ENGLAND SOUTH WEST     124259            57.3           3787                17


In [9]:
# ── SAVE GOLD TABLES ──────────────────────────────────────────────

# Provider level
path1 = os.path.join(GOLD_DIR, "gold_ae_provider_monthly.parquet")
gold_providers.to_parquet(path1, index=False)

# National time series
path2 = os.path.join(GOLD_DIR, "gold_ae_national_monthly.parquet")
national.to_parquet(path2, index=False)

# Regional
path3 = os.path.join(GOLD_DIR, "gold_ae_regional_monthly.parquet")
regional.to_parquet(path3, index=False)

print("✅ Gold layer complete")
print(f"  Provider monthly:  {len(gold_providers):,} rows — {os.path.getsize(path1)/1024:.1f} KB")
print(f"  National monthly:  {len(national):,} rows  — {os.path.getsize(path2)/1024:.1f} KB")
print(f"  Regional monthly:  {len(regional):,} rows  — {os.path.getsize(path3)/1024:.1f} KB")

✅ Gold layer complete
  Provider monthly:  7,238 rows — 319.0 KB
  National monthly:  36 rows  — 11.9 KB
  Regional monthly:  252 rows  — 18.9 KB


In [7]:
os.getcwd()

'/Users/yusufismail'